<a href="https://colab.research.google.com/github/onljunior72-dot/Proyecto/blob/main/Proyectoo3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start ColabSteam below</b><br/>
<audio autoplay="" src="https://github.com/kmille36/Colab-Cloud-Gaming/raw/refs/heads/main/silence.m4a" loop controls>

Importante una ves abierto la web
Configurar el bribate a 3mps
Y el audio a 64 kb/s y darle a la casilla de la casita para recargar
Para usarlo correcto pon en pantalla completa

In [26]:

# @title **🎮 Colab Virtual Desktop (Acelerado con T4 + Selkies + Cloudflare)**
# @markdown ---
# @markdown **🖼️ Wallpaper URL (opcional):**
background_url = ""  # @param {"type":"string"}

# @markdown **🌐 Website to auto-open (opcional):**
launch_url = ""  # @param {"type":"string"}

# @markdown **🖥️ Display Resolution:**
display_resolution = "1280x720"  # @param ["1280x720", "1366x768", "1600x900", "1920x1080"]

# @markdown **💾 Auto Backup on Exit (requiere montar Drive)**
auto_backup = False  # @param {"type":"boolean"}

import os, time, subprocess, re, glob, sys
from IPython.display import display, Javascript

# -----------------------------------------------------------------------------
# Funciones auxiliares
# -----------------------------------------------------------------------------
def run_cmd(cmd, check=True, silent=False):
    """Ejecuta un comando de shell."""
    if silent:
        subprocess.run(cmd, shell=True, check=check, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        subprocess.run(cmd, shell=True, check=check)

def print_step(msg):
    print("\n" + "="*60)
    print(f"👉 {msg}")
    print("="*60)

# -----------------------------------------------------------------------------
# Paso 0: Montar Drive si se activa backup
# -----------------------------------------------------------------------------
if auto_backup:
    try:
        from google.colab import drive
        print("⏳ Montando Google Drive...")
        drive.mount("/content/drive", force_remount=True)
        !mkdir -p /content/drive/MyDrive/colab_selkies_backup
    except Exception as e:
        print(f"⚠️ No se pudo montar Drive: {e}")

# -----------------------------------------------------------------------------
# Paso 1: Limpieza inicial
# -----------------------------------------------------------------------------
print_step("PASO 1: Limpiando procesos y reparando dpkg")
run_cmd("pkill -9 Xorg 2>/dev/null", check=False)
run_cmd("pkill -9 openbox 2>/dev/null", check=False)
run_cmd("pkill -9 selkies 2>/dev/null", check=False)
run_cmd("pkill -9 cloudflared 2>/dev/null", check=False)
run_cmd("rm -f /tmp/.X1-lock 2>/dev/null", check=False)
run_cmd("dpkg --configure -a", silent=True)
run_cmd("apt-get update -qq", silent=True)
print("✅ Listo.")

# -----------------------------------------------------------------------------
# Paso 2: Instalar paquetes del repositorio original (packages.tar.gz)
# -----------------------------------------------------------------------------
print_step("PASO 2: Instalando paquetes base desde el repositorio")
if not glob.glob("/packages/NVIDIA*.run"):
    run_cmd("rm -rf /packages")
    run_cmd("wget -qO- https://github.com/OkamuraYuji/Colab-Gaming/releases/download/1.0.0/packages.tar.gz | pigz -dc | tar -xv -C /")
    run_cmd("dpkg --add-architecture i386; apt update", silent=True)
    run_cmd("DEBIAN_FRONTEND=noninteractive dpkg -i /packages/*.deb 2>/dev/null || true", silent=True)
    run_cmd("DEBIAN_FRONTEND=noninteractive apt-get install --fix-broken -y -o Dir::Cache::archives='/packages'", silent=True)
    run_cmd("apt-get install -y -qq pv tint2 dbus-x11 net-tools", silent=True)
    run_cmd("chmod +x /packages/NVIDIA*.run && echo 1 | /packages/NVIDIA*.run --no-kernel-module --ui=none", silent=True)
print("✅ Paquetes instalados.")

# -----------------------------------------------------------------------------
# Paso 3: Configurar usuario y menú de Openbox
# -----------------------------------------------------------------------------
print_step("PASO 3: Configurando usuario y menú de Openbox")
run_cmd("echo 'root:123456' | sudo chpasswd", silent=True)
if not os.path.exists("/home/user"):
    run_cmd("sudo useradd -m -s /bin/bash user", silent=True)
    run_cmd("echo 'user:123456' | sudo chpasswd", silent=True)
    run_cmd("usermod -aG sudo user", silent=True)
if not os.path.exists("/home/user/wallpaper.jpg"):
    run_cmd("cp /packages/wallpaper.jpg /home/user/ 2>/dev/null || true", silent=True)
    run_cmd("chown user:user /home/user/wallpaper.jpg 2>/dev/null || true", silent=True)

# Crear menú de Openbox con los programas del backup
menu_dir = "/home/user/.config/openbox"
os.makedirs(menu_dir, exist_ok=True)
menu_xml = """<?xml version="1.0" encoding="UTF-8"?>
<openbox_menu xmlns="http://openbox.org/3.4/menu">
  <menu id="root-menu" label="Openbox 3">
    <item label="🌐 Google Chrome">
      <action name="Execute"><execute>google-chrome --no-sandbox --disable-gpu-sandbox --test-type</execute></action>
    </item>
    <item label="💻 Terminal Xfce">
      <action name="Execute"><execute>xfce4-terminal</execute></action>
    </item>
    <item label="📁 Gestor de archivos Thunar">
      <action name="Execute"><execute>thunar</execute></action>
    </item>
    <item label="🖼️ Visor de imágenes feh">
      <action name="Execute"><execute>feh</execute></action>
    </item>
    <separator />
    <item label="🔄 Recargar Openbox">
      <action name="Reconfigure" />
    </item>
    <item label="🚪 Salir">
      <action name="Exit" />
    </item>
  </menu>
</openbox_menu>"""
with open(f"{menu_dir}/menu.xml", "w") as f:
    f.write(menu_xml)
run_cmd(f"chown -R user:user {menu_dir}", check=False, silent=True)
print("✅ Usuario 'user' listo (contraseña: 123456). Menú de Openbox configurado.")

# -----------------------------------------------------------------------------
# Paso 4: Iniciar el servidor Xorg con aceleración GPU
# -----------------------------------------------------------------------------
print_step("PASO 4: Iniciando servidor gráfico Xorg (acelerado por T4)")
run_cmd("pkill -9 Xorg 2>/dev/null", check=False)
run_cmd("rm -f /tmp/.X1-lock 2>/dev/null", check=False)

# Configurar Xorg para usar la GPU NVIDIA y una pantalla virtual
run_cmd(f"nvidia-xconfig -a --allow-empty-initial-configuration --virtual={display_resolution} --busid PCI:0:4:0", silent=False)
# Iniciar el servidor Xorg en el display :1
subprocess.Popen(["nohup", "sudo", "Xorg", ":1", "-seat", "seat-1", "-allowMouseOpenFail", "-novtswitch", "-nolisten", "tcp"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

if subprocess.run("pgrep -f 'Xorg :1'", shell=True).returncode != 0:
    print("❌ Xorg no pudo iniciarse. Abortando.")
    raise RuntimeError("Xorg failed")
print("✅ Xorg corriendo en DISPLAY=:1")
os.environ['DISPLAY'] = ':1'
run_cmd("DISPLAY=:1 xhost +local:", silent=False)

print("✅ Servidor Xorg con aceleración T4 listo.")

# -----------------------------------------------------------------------------
# Paso 5: Instalar/Iniciar Openbox sobre el nuevo Xorg
# -----------------------------------------------------------------------------
print_step("PASO 5: Lanzando Openbox sobre Xorg")
if not os.path.exists("/usr/bin/openbox"):
    run_cmd("dpkg -i /packages/openbox*.deb 2>/dev/null; apt-get install -f -y -qq", silent=True)
subprocess.Popen(["nohup", "openbox", "--replace"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
if subprocess.run("pgrep -f 'openbox --replace'", shell=True).returncode != 0:
    print("⚠️ Openbox no se detectó, pero podría estar funcionando.")
else:
    print("✅ Openbox iniciado")

# Fondo de pantalla y panel tint2
if os.path.exists("/home/user/wallpaper.jpg") and os.path.exists("/usr/bin/feh"):
    subprocess.Popen(["nohup", "feh", "--bg-max", "/home/user/wallpaper.jpg"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists("/usr/bin/tint2"):
    subprocess.Popen(["nohup", "tint2"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("✅ Escritorio Openbox listo sobre Xorg acelerado.")

# -----------------------------------------------------------------------------
# Paso 6: Descargar Selkies y Cloudflared
# -----------------------------------------------------------------------------
print_step("PASO 6: Obteniendo Selkies y Cloudflared")
selkies_dirs = glob.glob('/content/*selkies*')
if not selkies_dirs:
    print("⬇️ Descargando Selkies...")
    run_cmd("wget -q --show-progress https://github.com/selkies-project/selkies/releases/download/v1.6.2/selkies-gstreamer-portable-v1.6.2_amd64.tar.gz -O /content/selkies.tar.gz")
    run_cmd("tar -xzf /content/selkies.tar.gz -C /content/")
    run_cmd("rm /content/selkies.tar.gz")
    selkies_dirs = glob.glob('/content/*selkies*')
    if selkies_dirs:
        print(f"✅ Selkies extraído en: {selkies_dirs[0]}")
else:
    print(f"✅ Selkies ya está en: {selkies_dirs[0]}")
if not os.path.exists('/content/cloudflared'):
    print("⬇️ Descargando cloudflared...")
    run_cmd("wget -q --show-progress https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared")
    run_cmd("chmod +x /content/cloudflared")
    print("✅ cloudflared descargado.")
else:
    print("✅ cloudflared ya está presente.")

# -----------------------------------------------------------------------------
# Paso 7: Iniciar Selkies con soporte para NVENC (aceleración de video)
# -----------------------------------------------------------------------------
print_step("PASO 7: Iniciando Selkies (con NVENC) en puerto 8081")
run_cmd("pkill -9 selkies 2>/dev/null", check=False)
selkies_dirs = glob.glob('/content/*selkies*')
if not selkies_dirs:
    raise Exception("No se encontró la carpeta de Selkies.")
selkies_dir = selkies_dirs[0]
executable = None
for root, dirs, files in os.walk(selkies_dir):
    for f in files:
        if 'selkies-gstreamer-run' in f or f == 'selkies-gstreamer-run':
            executable = os.path.join(root, f)
            break
    if executable:
        break
if not executable:
    raise Exception(f"No se encontró el ejecutable de Selkies dentro de {selkies_dir}")
print(f"🚀 Ejecutable: {executable}")
with open('/content/selkies.log', 'w') as log:
    subprocess.Popen([executable, '--port', '8081',
                      '--basic_auth_user', 'user', '--basic_auth_password', '123456',
                      '--enable_resize', 'false', '--encoder', 'nvh264enc'],  # Se fuerza el uso de NVENC
                     stdout=log, stderr=subprocess.STDOUT, cwd=os.path.dirname(executable))
time.sleep(5)
if subprocess.run("netstat -tlnp 2>/dev/null | grep 8081", shell=True).returncode == 0:
    print("✅ Selkies escuchando en puerto 8081 (con codificación NVENC)")
else:
    print("⚠️ Selkies podría no haberse iniciado. Revisa /content/selkies.log")

# -----------------------------------------------------------------------------
# Paso 8: Iniciar Cloudflare y extraer URL
# -----------------------------------------------------------------------------
print_step("PASO 8: Iniciando túnel Cloudflare")
run_cmd("pkill -9 cloudflared 2>/dev/null", check=False)
run_cmd("echo '' > /content/tunnel.log", silent=True)
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8081'],
                 stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
print("⏳ Esperando enlace...")
url_encontrada = None
for i in range(30):
    time.sleep(1.5)
    try:
        with open('/content/tunnel.log', 'r') as f:
            log = f.read()
            match = re.search(r"https://[-a-zA-Z0-9.]*\.trycloudflare\.com", log)
            if match:
                url_encontrada = match.group(0)
                break
    except:
        pass
if url_encontrada:
    print("\n" + "🎉"*30)
    print("🎉 ¡ESCRITORIO ACELERADO LISTO!")
    print("🎉"*30)
    print(f"\n🌐 ENLACE: {url_encontrada}")
    print(f"👤 USUARIO: user")
    print(f"🔑 CONTRASEÑA: 123456")
    print("\n💡 La renderización 3D y la codificación de video están aceleradas por la GPU T4.")
else:
    print("\n⚠️ No se pudo obtener la URL. Revisa /content/tunnel.log")

# -----------------------------------------------------------------------------
# Paso 9: Abrir Chrome si se especificó URL
# -----------------------------------------------------------------------------
if launch_url:
    print_step("PASO 9: Abriendo navegador con URL")
    chrome_cmd = f"DISPLAY=:1 google-chrome --no-sandbox --disable-gpu-sandbox --test-type --start-maximized {launch_url} &"
    run_cmd(chrome_cmd, check=False)
    print(f"✅ Chrome lanzado con {launch_url}")

# -----------------------------------------------------------------------------
# Keep‑Alive de Colab
# -----------------------------------------------------------------------------
print_step("MANTENIENDO SESIÓN ACTIVA")
keep_alive_js = """
function keepAlive() {
    var btn = document.querySelector('#top-toolbar > colab-connect-button');
    if (btn) { btn.click(); console.log('Keep-alive: click'); }
    setTimeout(keepAlive, 60000);
}
keepAlive();
"""
display(Javascript(keep_alive_js))
print("🔄 Keep‑Alive activado. La sesión no se cerrará por inactividad.")
print("\n✅ Proceso completado. Disfruta tu escritorio virtual acelerado por GPU.")


👉 PASO 1: Limpiando procesos y reparando dpkg
✅ Listo.

👉 PASO 2: Instalando paquetes base desde el repositorio
✅ Paquetes instalados.

👉 PASO 3: Configurando usuario y menú de Openbox
✅ Usuario 'user' listo (contraseña: 123456). Menú de Openbox configurado.

👉 PASO 4: Iniciando servidor gráfico Xorg (acelerado por T4)
✅ Xorg corriendo en DISPLAY=:1
✅ Servidor Xorg con aceleración T4 listo.

👉 PASO 5: Lanzando Openbox sobre Xorg
✅ Openbox iniciado
✅ Escritorio Openbox listo sobre Xorg acelerado.

👉 PASO 6: Obteniendo Selkies y Cloudflared
✅ Selkies ya está en: /content/selkies-gstreamer
✅ cloudflared ya está presente.

👉 PASO 7: Iniciando Selkies (con NVENC) en puerto 8081
🚀 Ejecutable: /content/selkies-gstreamer/selkies-gstreamer-run
✅ Selkies escuchando en puerto 8081 (con codificación NVENC)

👉 PASO 8: Iniciando túnel Cloudflare
⏳ Esperando enlace...

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
🎉 ¡ESCRITORIO ACELERADO LISTO!
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉

🌐 ENLACE: https://many-unto-contrast-a

<IPython.core.display.Javascript object>

🔄 Keep‑Alive activado. La sesión no se cerrará por inactividad.

✅ Proceso completado. Disfruta tu escritorio virtual acelerado por GPU.


In [21]:
# @title 💻 Abrir terminal Xfce en el escritorio virtual
import subprocess
import os

# Asegurar que apuntamos al display correcto
os.environ['DISPLAY'] = ':1'

# Intentar con xfce4-terminal (más completo), si no existe usamos xterm
try:
    subprocess.Popen(['xfce4-terminal'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ Terminal Xfce abierta en el escritorio remoto.")
except FileNotFoundError:
    subprocess.Popen(['xterm'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ Terminal xterm abierta (xfce4-terminal no encontrada).")

✅ Terminal Xfce abierta en el escritorio remoto.


In [22]:
# @title 🌐 Abrir Google Chrome en el escritorio virtual
import subprocess
import os
import time

# 1. Asegurar que el display virtual está definido
os.environ['DISPLAY'] = ':1'

# 2. Comando para lanzar Chrome (sin sandbox, sin dbus)
chrome_cmd = [
    'google-chrome-stable',
    '--no-sandbox',
    '--disable-gpu-sandbox',
    '--test-type',
    '--start-maximized',
    '--disable-features=UseDbus'   # Evita errores de D-Bus
]

print("🚀 Abriendo Google Chrome en el escritorio remoto...")
# Ejecutar Chrome en segundo plano
proc = subprocess.Popen(chrome_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

# Verificar si se lanzó correctamente
if proc.poll() is None:
    print("✅ Chrome se está ejecutando. Deberías verlo en la ventana de Selkies.")
else:
    print("❌ Chrome no pudo abrirse. Puede que falte instalarlo.")
    print("   Para instalarlo ejecuta primero la celda de instalación de Chrome.")

🚀 Abriendo Google Chrome en el escritorio remoto...
✅ Chrome se está ejecutando. Deberías verlo en la ventana de Selkies.


In [19]:
# Celda para iniciar Cloudflare (si no se ve el enlace)
import subprocess, time, re
subprocess.run("pkill -9 cloudflared", shell=True)
subprocess.run("echo '' > /content/tunnel.log", shell=True)
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8081"],
                 stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
with open('/content/tunnel.log', 'r') as f:
    match = re.search(r"https://[-a-zA-Z0-9.]*\.trycloudflare\.com", f.read())
    if match:
        print(f"🌐 ENLACE: {match.group(0)}")

🌐 ENLACE: https://certified-quarters-medicaid-corps.trycloudflare.com


In [ ]:
# @title 🔄 **Keep-Alive con Python (bucle infinito)**
import time
from IPython.display import clear_output

print("⏳ Keep‑Alive activado. Presiona el botón ■ 'Interrumpir ejecución' para detenerlo.")
i = 0
while True:
    i += 1
    clear_output(wait=True)
    print(f"🔄 Keep‑Alive: sesión activa [{i}] - {time.strftime('%H:%M:%S')}")
    time.sleep(60)  # imprime cada 60 segundos

🔄 Keep‑Alive: sesión activa [6] - 22:11:52
